<a href="https://colab.research.google.com/github/Chandrani45/AI-COURSE-RECOMMENDER-PROJECT-WORK-/blob/main/Chandrani_Sengupta_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

LOADING THE LIBRARIES

In [9]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [10]:
!pip install groq

MODULE 0
Loading Groq

In [11]:
import os
from groq import Groq

In [12]:
import os
from getpass import getpass

os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key: ")

Enter your Groq API key: ··········


In [13]:
client = Groq()

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "system", "content": "You are a concise assistant."},
        {"role": "user", "content": "Explain why Groq's inference is fast in 3 sentences."},
    ],
)

print(response.choices[0].message.content)

Groq's inference is fast due to its tensor processing unit (TPU) architecture, which is designed specifically for machine learning workloads. The TPU's bespoke design allows for highly efficient processing of tensor operations, resulting in improved performance and reduced latency. Additionally, Groq's compiler technology optimizes models for the TPU, further enhancing inference speed and making it suitable for high-performance applications.


LOADING THE COURSE DATASET

In [14]:
course=pd.read_csv('/content/coursera_course_dataset_v3.csv')

PRINTING THE FIRST 5 ROWS

In [15]:
course.head()

,Unnamed: 0,Title,Organization,Skills,Ratings,course_url,course_students_enrolled,course_description,Review Count,Difficulty,Type,Duration
0,0,Google Cybersecurity,Google,"Network Security, Python Programming, Linux, ...",4.8,https://www.coursera.org/professional-certific...,"700,909",Google Cloud Fundamentals: Core Infrastructure...,20K,Beginner,Professional Certificate,3 - 6 Months
1,1,Google Data Analytics,Google,"Data Analysis, R Programming, SQL, Business C...",4.8,https://www.coursera.org/professional-certific...,"229,865",Prepare for a new career in the high-growth fi...,137K,Beginner,Professional Certificate,3 - 6 Months
2,3,Google Project Management:,Google,"Project Management, Strategy and Operations, ...",4.8,https://www.coursera.org/professional-certific...,"29,702",Prepare-se para uma nova carreira no campo de ...,100K,Beginner,Professional Certificate,3 - 6 Months
3,4,IBM Data Science,IBM,"Python Programming, Data Science, Machine Lea...",4.6,https://www.coursera.org/professional-certific...,"239,622",Prepare for a career in the high-growth field ...,120K,Beginner,Professional Certificate,3 - 6 Months
4,5,Google Digital Marketing & E-commerce,Google,"Digital Marketing, Marketing, Marketing Manag...",4.8,https://www.coursera.org/professional-certific...,"384,238",This course is the eighth course in the Google...,23K,Beginner,Professional Certificate,3 - 6 Months


We drop the unnamed column as it is of no use

In [16]:
course.rename(columns={'Unnamed: 0':'Course_id'},inplace=True)

In [17]:
course.head()

,Course_id,Title,Organization,Skills,Ratings,course_url,course_students_enrolled,course_description,Review Count,Difficulty,Type,Duration
0,0,Google Cybersecurity,Google,"Network Security, Python Programming, Linux, ...",4.8,https://www.coursera.org/professional-certific...,"700,909",Google Cloud Fundamentals: Core Infrastructure...,20K,Beginner,Professional Certificate,3 - 6 Months
1,1,Google Data Analytics,Google,"Data Analysis, R Programming, SQL, Business C...",4.8,https://www.coursera.org/professional-certific...,"229,865",Prepare for a new career in the high-growth fi...,137K,Beginner,Professional Certificate,3 - 6 Months
2,3,Google Project Management:,Google,"Project Management, Strategy and Operations, ...",4.8,https://www.coursera.org/professional-certific...,"29,702",Prepare-se para uma nova carreira no campo de ...,100K,Beginner,Professional Certificate,3 - 6 Months
3,4,IBM Data Science,IBM,"Python Programming, Data Science, Machine Lea...",4.6,https://www.coursera.org/professional-certific...,"239,622",Prepare for a career in the high-growth field ...,120K,Beginner,Professional Certificate,3 - 6 Months
4,5,Google Digital Marketing & E-commerce,Google,"Digital Marketing, Marketing, Marketing Manag...",4.8,https://www.coursera.org/professional-certific...,"384,238",This course is the eighth course in the Google...,23K,Beginner,Professional Certificate,3 - 6 Months


In [18]:
course.columns

Index(['Course_id', 'Title', 'Organization', 'Skills', 'Ratings', 'course_url',
       'course_students_enrolled', 'course_description', 'Review Count',
       'Difficulty', 'Type', 'Duration'],
      dtype='object')

In [19]:
course.columns=course.columns.str.lower()


In [20]:
course.head()

,course_id,title,organization,skills,ratings,course_url,course_students_enrolled,course_description,review count,difficulty,type,duration
0,0,Google Cybersecurity,Google,"Network Security, Python Programming, Linux, ...",4.8,https://www.coursera.org/professional-certific...,"700,909",Google Cloud Fundamentals: Core Infrastructure...,20K,Beginner,Professional Certificate,3 - 6 Months
1,1,Google Data Analytics,Google,"Data Analysis, R Programming, SQL, Business C...",4.8,https://www.coursera.org/professional-certific...,"229,865",Prepare for a new career in the high-growth fi...,137K,Beginner,Professional Certificate,3 - 6 Months
2,3,Google Project Management:,Google,"Project Management, Strategy and Operations, ...",4.8,https://www.coursera.org/professional-certific...,"29,702",Prepare-se para uma nova carreira no campo de ...,100K,Beginner,Professional Certificate,3 - 6 Months
3,4,IBM Data Science,IBM,"Python Programming, Data Science, Machine Lea...",4.6,https://www.coursera.org/professional-certific...,"239,622",Prepare for a career in the high-growth field ...,120K,Beginner,Professional Certificate,3 - 6 Months
4,5,Google Digital Marketing & E-commerce,Google,"Digital Marketing, Marketing, Marketing Manag...",4.8,https://www.coursera.org/professional-certific...,"384,238",This course is the eighth course in the Google...,23K,Beginner,Professional Certificate,3 - 6 Months


In [21]:
#print rows and columns
course.shape

(623, 12)

The data has 623 rows and 11 columns

In [22]:
role_skills={'Data Scientist': ['python', 'statistics', 'machine learning', 'sql',
                           'data visualization', 'deep learning'], 'Data Analyst': ['excel', 'sql', 'statistics', 'data visualization', 'python'], 'ML Engineer':['python', 'machine learning', 'deep learning', 'sql', 'cloud'],'Data Engineer': ['python','sql', 'data wrangling', 'cloud', 'big data', 'apis'], 'Business Analyst': ['excel', 'sql', 'statistics', 'data visualization',
                           'communication'],'BI Developer': ['sql', 'data visualization', 'excel', 'statistics', 'power bi'],'AI Researcher':['python', 'deep learning', 'machine learning',
                           'mathematics', 'nlp'], 'Backend Developer': ['python', 'sql', 'apis', 'cloud', 'git'],'MLOps Engineer':['python', 'machine learning', 'cloud', 'docker', 'mlops'], 'NLP Engineer':['python', 'machine learning', 'deep learning', 'nlp',
                           'statistics']



}

In [23]:
role_skills['Data Scientist']

['python',
 'statistics',
 'machine learning',
 'sql',
 'data visualization',
 'deep learning']

In [24]:
course.isnull().sum()

,0
course_id,0
title,0
organization,0
skills,0
ratings,0
course_url,220
course_students_enrolled,236
course_description,221
review count,0
difficulty,0


In [25]:
course.duplicated().sum()

np.int64(0)

No duplicates as such

In [26]:
course.drop(columns=['organization','ratings','course_url' ,	'course_students_enrolled' ,'course_description', 'review count','difficulty','type','duration'],axis=1,inplace=True)

In [27]:
course.isnull().sum()

,0
course_id,0
title,0
skills,0


In [28]:
course.head()

,course_id,title,skills
0,0,Google Cybersecurity,"Network Security, Python Programming, Linux, ..."
1,1,Google Data Analytics,"Data Analysis, R Programming, SQL, Business C..."
2,3,Google Project Management:,"Project Management, Strategy and Operations, ..."
3,4,IBM Data Science,"Python Programming, Data Science, Machine Lea..."
4,5,Google Digital Marketing & E-commerce,"Digital Marketing, Marketing, Marketing Manag..."


In [29]:
course.shape

(623, 3)

In [30]:
#cleaning the skills dataset
import re
def clean(text):
  return re.sub(r'[^a-z0-9]','',text.str.lower())
  return re.sub(r'\s+',"",text).strip()

In [31]:
import re

def clean(s):
    s = re.sub(r"[^a-z0-9 ]", " ", str(s).lower())
    return re.sub(r"\s+", " ", s).strip()

# build the search text from Title + Skills  (note the " " between them)
course['text'] = (course['title'] + " " + course['skills']).apply(clean)

# keep a clean LIST of skills — M6 needs this
course['skills'] = course['skills'].fillna("").apply(
    lambda s: [x.strip().lower() for x in str(s).split(",") if x.strip()])

course[['course_id', 'title', 'skills', 'text']].head(3)

,course_id,title,skills,text
0,0,Google Cybersecurity,"[network security, python programming, linux, ...",google cybersecurity network security python p...
1,1,Google Data Analytics,"[data analysis, r programming, sql, business c...",google data analytics data analysis r programm...
2,3,Google Project Management:,"[project management, strategy and operations, ...",google project management project management s...


In [32]:
course.head()

,course_id,title,skills,text
0,0,Google Cybersecurity,"[network security, python programming, linux, ...",google cybersecurity network security python p...
1,1,Google Data Analytics,"[data analysis, r programming, sql, business c...",google data analytics data analysis r programm...
2,3,Google Project Management:,"[project management, strategy and operations, ...",google project management project management s...
3,4,IBM Data Science,"[python programming, data science, machine lea...",ibm data science python programming data scien...
4,5,Google Digital Marketing & E-commerce,"[digital marketing, marketing, marketing manag...",google digital marketing e commerce digital ma...


In [33]:
course['text'].iloc[0]

'google cybersecurity network security python programming linux cloud computing algorithms audit computer programming computer security incident management cryptography databases leadership and management network architecture risk management sql'

In [34]:
target_role = input('Enter a role: ').strip()
assert target_role in role_skills, f"'{target_role}' is not in the role list. Pick one of: {list(role_skills)}"

Enter a role: Data Scientist


In [35]:
target_role

'Data Scientist'

In [36]:
current_skills = [s.strip().lower() for s in input('Enter your current skills (comma-separated): ').split(",") if s.strip()]
have = set(current_skills)


Enter your current skills (comma-separated): python


In [37]:
current_skills

['python']

In [38]:
skill_gap = [s for s in role_skills[target_role] if s.lower() not in have]
query_text = " ".join(skill_gap)
print("skill_gap:", skill_gap)

skill_gap: ['statistics', 'machine learning', 'sql', 'data visualization', 'deep learning']


In [39]:
query_text

'statistics machine learning sql data visualization deep learning'

In [40]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [41]:
tfidf=TfidfVectorizer(stop_words='english')

In [65]:
course_vector=tfidf.fit_transform(course['text']).toarray()

In [43]:
from sklearn.metrics.pairwise import cosine_similarity

In [44]:
def recommend(query_text):
  query_vector=tfidf.transform([query_text])
  similarity=cosine_similarity(course_vector,query_vector).flatten()
  course_list=course.copy()
  course_list['score']=similarity
  course_list=course_list.sort_values(by='score',ascending=False)
  return course_list['title'].head(10)

In [45]:
result=recommend(query_text)

In [46]:
result

,title
194,Machine Learning on Google Cloud
7,Machine Learning
93,IBM Machine Learning
519,Applying Machine Learning to your Data with Go...
288,Deep Neural Networks with PyTorch
408,Introduction to Artificial Intelligence (AI)
149,"Unsupervised Learning, Recommenders, Reinforce..."
87,Advanced Learning Algorithms
39,IBM AI Engineering
100,IBM Introduction to Machine Learning


In [47]:
#embedding
!pip install gensim


In [48]:
import gensim

In [49]:
import os

In [63]:
model=gensim.models.Word2Vec(window=10,min_count=2,workers=4,vector_size=100, sorted_vocab=False)

In [55]:
model.build_vocab(course['text'].apply(lambda x: x.split()))

In [52]:
model.train(course['text'].apply(lambda x: x.split()), total_examples=model.corpus_count, epochs=model.epochs)

(49055, 88610)

In [80]:
def document_vector(query_text):
  query_vector=[word for word in query_text.split() if word in model.wv.index_to_key]
  for w in query_vector:
    return model.wv[w]


In [81]:
que_vec=document_vector(query_text)

In [82]:
que_vec

In [54]:
model.wv['python']

array([-0.10627463, -0.07044705,  0.04529295, -0.11756142, -0.15475056,
       -0.37355226,  0.18525353,  0.5158782 , -0.2909732 , -0.48353863,
       -0.09276991, -0.2920246 ,  0.02828341,  0.04199214,  0.09844884,
       -0.17650062,  0.00645035, -0.41493782, -0.26869425, -0.5114231 ,
       -0.01887296,  0.10658345,  0.10489769, -0.17663027, -0.09252502,
       -0.00121421, -0.12716934, -0.10130277, -0.1724386 ,  0.15989576,
        0.06164763, -0.03031665,  0.02222918, -0.15476991, -0.15305892,
        0.42857397,  0.15368089, -0.26875544, -0.20566125, -0.30142832,
        0.08994085, -0.3336296 , -0.12686971, -0.22593331,  0.2826698 ,
       -0.20292833, -0.23712818, -0.11927246,  0.15466362,  0.1328835 ,
        0.05915296, -0.23819633,  0.17535767, -0.03868826, -0.12884386,
        0.2742062 ,  0.12986577,  0.07738087, -0.12043042,  0.24197891,
        0.08195004, -0.04759968,  0.04589682, -0.25768337, -0.4044916 ,
        0.3384048 , -0.05547792,  0.19740364, -0.12733461,  0.34